<a href="https://colab.research.google.com/github/NghiaH19/E-commerce/blob/main/TrainingLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

path = '/content/drive/MyDrive/Project2_Ecommerce/ecommerce_final_processed.csv'
df = pd.read_csv(path)

df['date'] = pd.to_datetime(df['date'])
print(f"✅ Đã nạp {len(df):,} dòng dữ liệu sẵn sàng huấn luyện!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã nạp 2,662,922 dòng dữ liệu sẵn sàng huấn luyện!


In [ ]:
split_date = '2019-11-24'

train = df[df['date'] < split_date]
test = df[df['date'] >= split_date]

features = ['price', 'day_of_week', 'is_weekend', 'view_lag_1', 'view_lag_7', 'mua_lag_1', 'view_rolling_7', ]
target = 'luot_mua'

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print(f"Tập Học (Train): {len(X_train):,} dòng")
print(f"Tập Thi (Test): {len(X_test):,} dòng")

Tập Học (Train): 2,352,029 dòng
Tập Thi (Test): 310,893 dòng


In [ ]:
split_date = '2019-11-24'

train = df[df['date'] < split_date]
test = df[df['date'] >= split_date]

features = ['price', 'day_of_week', 'is_weekend', 'view_lag_1', 'view_lag_7', 'mua_lag_1', 'view_rolling_7', 'category_code', 'brand']
target = 'luot_mua'

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print(f"Tập Học (Train): {len(X_train):,} dòng")
print(f"Tập Thi (Test): {len(X_test):,} dòng")

Tập Học (Train): 2,352,029 dòng
Tập Thi (Test): 310,893 dòng


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled = scaler_x.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

print(f"✅ Đã chuẩn hóa và chuyển đổi xong!")
print(f"Hình dáng mới của X_train: {X_train_lstm.shape}")

✅ Đã chuẩn hóa và chuyển đổi xong!
Hình dáng mới của X_train: (2352029, 1, 9)


In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. Khởi tạo mô hình tuần tự
model_lstm = Sequential([
    # Tầng LSTM đầu tiên: 64 đơn vị
    LSTM(64, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2]), return_sequences=True),
    Dropout(0.2), # Bỏ qua ngẫu nhiên 20% neuron để chống học vẹt

    LSTM(32, activation='relu'),
    Dropout(0.2),

    Dense(16, activation='relu'),

    Dense(1)
])

# 2. Biên dịch mô hình
model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

model_lstm.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 1, 64)          │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,905 (124.63 KB)

 Trainable params: 31,905 (124.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Cài đặt phanh khẩn cấp
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("Hệ thống đang bắt đầu huấn luyện!")

# Bắt đầu Train
history = model_lstm.fit(
    X_train_lstm, y_train_scaled,
    epochs=100,             # Cho AI học tối đa 30 lần
    batch_size=2048,        # Xử lý 2048 dòng cùng lúc để tận dụng GPU
    validation_data=(X_test_lstm, y_test_scaled),
    callbacks=[early_stop],
    verbose=1
)

Hệ thống đang bắt đầu huấn luyện!
Epoch 1/100
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 7.3515e-07 - mae: 1.1023e-04 - val_loss: 2.9548e-07 - val_mae: 1.4049e-04
Epoch 2/100
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 5.6241e-07 - mae: 1.0883e-04 - val_loss: 3.7206e-07 - val_mae: 1.0791e-04
Epoch 3/100
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 7.5797e-07 - mae: 1.1630e-04 - val_loss: 5.2019e-07 - val_mae: 5.8220e-05
Epoch 4/100
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 6.4650e-07 - mae: 1.1694e-04 - val_loss: 3.1964e-07 - val_mae: 6.9043e-05
Epoch 5/100
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 6.5428e-07 - mae: 1.1786e-04 - val_loss: 2.7075e-07 - val_mae: 5.3952e-05
Epoch 6/100
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 5.5245e-07 - mae: 1.0565e-04 - val_loss: 3.7615e-07 - val_mae: 7.8026e-05
Epoch 7/100
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 5.7577e-07 - mae: 1.1246e-04 - val_loss: 4.7428e-07 - val_mae: 1.7775e-04
Epoch 8

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Dự báo và giải mã
y_pred_scaled = model_lstm.predict(X_test_lstm)
y_pred_final = scaler_y.inverse_transform(y_pred_scaled)
y_test_final = scaler_y.inverse_transform(y_test_scaled)
y_pred_final = np.maximum(y_pred_final, 0)

# 2. Tính toán bộ 3 chỉ số "vàng"
mae_lstm = mean_absolute_error(y_test_final, y_pred_final)
mse_lstm = mean_squared_error(y_test_final, y_pred_final)
rmse_lstm = np.sqrt(mse_lstm)
r2_lstm = r2_score(y_test_final, y_pred_final)

print(f"KẾT QUẢ CỦA LSTM")
print(f"✅ MAE  : {mae_lstm:.4f}")
print(f"✅ RMSE : {rmse_lstm:.4f}")
print(f"✅ R2   : {r2_lstm:.4f}")

9716/9716 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step
KẾT QUẢ CỦA LSTM
✅ MAE  : 0.2319
✅ RMSE : 2.1905
✅ R2   : 0.9390


In [ ]:
import joblib
import os

# Đường dẫn thư mục lưu trữ
save_dir = '/content/drive/MyDrive/Project2_Ecommerce/'

# 1. Lưu mô hình LSTM
model_lstm.save(os.path.join(save_dir, 'lstm_demand_model.keras'))

# 2. Lưu bộ chuẩn hóa
joblib.dump(scaler_x, os.path.join(save_dir, 'scaler_x_lstm.pkl'))
joblib.dump(scaler_y, os.path.join(save_dir, 'scaler_y_lstm.pkl'))

print("✅ Đã trích xuất thành công bộ 3: Model, Scaler_X, và Scaler_Y lên Drive!")

✅ Đã trích xuất thành công bộ 3: Model, Scaler_X, và Scaler_Y lên Drive!
